#The Transformation Logic

In [0]:
query = """
SELECT 
ROW_NUMBER() OVER (ORDER BY c.customer_id) AS customer_key, -- Surrogate Key
c.customer_id,
c.customer_number,
c.customer_firstname,
c.customer_lastname,
cl.country,
c.customer_marital_status,
CASE
    WHEN c.customer_gender <> 'n/a' THEN c.customer_gender
    ELSE COALESCE(c.customer_gender, 'n/a')
END gender,
ca.birth_date,
c.created_date
FROM silver.crm_customers c
LEFT JOIN silver.erp_customer ca ON c.customer_number = ca.customer_number
LEFT JOIN silver.erp_customer_location cl ON c.customer_number = cl.customer_number
"""
df = spark.sql(query)


##Sanity Check

In [0]:
df.limit(10).display()

customer_key,customer_id,customer_number,customer_firstname,customer_lastname,country,customer_marital_status,gender,birth_date,created_date
1,null,A01Ass,null,null,null,n/a,n/a,null,null
2,null,13451235,null,null,null,n/a,n/a,null,null
3,null,SF566,null,null,null,n/a,n/a,null,null
4,null,PO25,null,null,null,n/a,n/a,null,null
5,11000,AW00011000,Jon,Yang,Australia,Married,Male,1971-10-06,2025-10-06
6,11001,AW00011001,Eugene,Huang,Australia,Single,Male,1976-05-10,2025-10-06
7,11002,AW00011002,Ruben,Torres,Australia,Married,Male,1971-02-09,2025-10-06
8,11003,AW00011003,Christy,Zhu,Australia,Single,Female,1973-08-14,2025-10-06
9,11004,AW00011004,Elizabeth,Johnson,Australia,Single,Female,1979-08-05,2025-10-06
10,11005,AW00011005,Julio,Ruiz,Australia,Single,Male,1976-08-01,2025-10-06


#Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customer")

##Sanity Check of Gold table

In [0]:
%sql
SELECT * FROM gold.dim_customer LIMIT(10)

customer_key,customer_id,customer_number,customer_firstname,customer_lastname,country,customer_marital_status,gender,birth_date,created_date
1,null,SF566,null,null,null,n/a,n/a,null,null
2,null,PO25,null,null,null,n/a,n/a,null,null
3,null,13451235,null,null,null,n/a,n/a,null,null
4,null,A01Ass,null,null,null,n/a,n/a,null,null
5,11000,AW00011000,Jon,Yang,Australia,Married,Male,1971-10-06,2025-10-06
6,11001,AW00011001,Eugene,Huang,Australia,Single,Male,1976-05-10,2025-10-06
7,11002,AW00011002,Ruben,Torres,Australia,Married,Male,1971-02-09,2025-10-06
8,11003,AW00011003,Christy,Zhu,Australia,Single,Female,1973-08-14,2025-10-06
9,11004,AW00011004,Elizabeth,Johnson,Australia,Single,Female,1979-08-05,2025-10-06
10,11005,AW00011005,Julio,Ruiz,Australia,Single,Male,1976-08-01,2025-10-06
